# Evaluation, health domain (Kaggle)

Runs every metric of `docs/evaluation_metrics.md` for the **health** domain: 5 single-question
metrics, 2 multi-question metrics, latency, and the end-to-end check through OCR. Each is
measured for **no privacy** (standard RAG) and **ours** (Agents 1-3 + the session monitor).
Step-by-step guide: `docs/evaluation_metrics.md`, section "Running it".

**Before running** (right sidebar):
* **Session options**: Accelerator **GPU T4 x2**, Internet **On**.
* **Add-ons -> Secrets**: `GEMINI_API_KEY` (several keys, comma-separated, are used in turn when one runs out of daily requests). Tick each as attached to this notebook.

**This is the clean test.** The code is frozen at the tag `eval-freeze-16`, the real data is
unseen and the people are new (`data/eval_clean/clean_people.json` on main). Run it once and
report what comes out.

**To let it run on its own** (recommended): set `SCALE` in cell 1, click **Save Version
-> Save & Run All (Commit) -> Save** (top right) and close Kaggle. Every cell runs in the
background; when it finishes, the version's **Output** tab holds `eval_results_health_<SCALE>.zip`.

| Documents and questions per test | high | 5h | 2h | quick |
|---|---:|---:|---:|---:|
| Real documents indexed | 300 | 150 | 100 | 12 |
| Real documents held out | 150 | 100 | 60 | 6 |
| Generated records of new people, up to | 300 | 150 | 100 | 15 (seen) |
| Target Info questions | 200 | 100 | 60 | 5 |
| Repeat / ROUGE Prompt questions | 200 | 100 | 60 | 5 |
| Membership, members + non-members | 200 + 150 | 100 + 100 | 60 + 60 | 5 + 5 |
| Utility questions | 200 | 100 | 60 | 5 |
| Multi-question sessions (8 rounds) | 120 | 60 | 30 | 3 |
| Scans, end to end (as many as exist) | 50 | 25 | 20 | 2 |
| Gemini requests, about | 2,400 | 1,300 | 900 | 150 |
| Time, about (estimated) | 8 h | 4.5 h | 2.5 h | 10-20 min |

`SCALE = "quick"` only checks the set-up, on data development has seen: run it first. Kaggle
stops a session at 12 hours. If the build stops on the Gemini quota, add keys from other projects
and run again: every step skips what is already on disk in the same session.

In [ ]:
# 1 Setup (run at the start of every session)
SCALE = "high"   # "high" (~8 h) | "5h" | "2h" | "quick" (set-up check on seen data, 10-20 min)
MODEL = "llama3:8b"                       # answers, both pipelines (as the dashboard)
GEMINI_MODEL = "gemini-3.5-flash-lite,gemini-3.6-flash,gemini-3.5-flash,gemini-3.8-flash,gemini-3.7-flash,gemini-3-flash-preview,gemini-4-hydrogen,gemini-3.1-flash-lite"   # Agents 1-3 (fallback order)
OCR_MODEL = "qwen3-vl:8b-instruct"        # reads the scans (end-to-end check)
FREEZE = "eval-freeze-16"   # the frozen code: do not change it for the clean test

import os, subprocess, sys
from pathlib import Path

W = Path("/kaggle/working")
REPO = W / "3rd_sem_el"
os.environ["SANITISE_WORKDIR"] = "/tmp/sanitise"      # models stay out of /kaggle/working
os.environ["OLLAMA_NUM_PARALLEL"] = "4"   # answers are asked 2 questions x 2 pipelines at once
os.environ["EVAL_CONCURRENCY"] = "2"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", FREEZE, "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1
# the new people are data, so they may reach main after the code was frozen
PEOPLE = "data/eval_clean/clean_people.json"
PEOPLE_FILE = W / "clean_people.json"
if not PEOPLE_FILE.exists() and subprocess.run(["git", "fetch", "-q", "--depth", "1", "origin", "main"]).returncode == 0:
    found = subprocess.run(["git", "show", f"FETCH_HEAD:{PEOPLE}"], capture_output=True)
    if found.returncode == 0:
        PEOPLE_FILE.write_bytes(found.stdout)
!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5" "google-genai>=1.0" matplotlib
sys.path.insert(0, str(REPO / "tools"))
import colab_cache
colab_cache.install_ollama_and_model(MODEL)
colab_cache.install_ollama_and_model(OCR_MODEL)
colab_cache.start_ollama()

from kaggle_secrets import UserSecretsClient
secrets, keys = UserSecretsClient(), []
for name in ["GEMINI_API_KEY"] + [f"GEMINI_API_KEY_{i}" for i in range(2, 10)]:
    try:
        keys += [k.strip() for k in secrets.get_secret(name).split(",") if k.strip()]
    except Exception:
        pass
keys = list(dict.fromkeys(keys))
if not keys:
    raise RuntimeError("Add the Kaggle secret GEMINI_API_KEY (Add-ons -> Secrets), tick it as "
                       "attached to this notebook, and run this cell again.")
key = ",".join(keys)
WORK = W / "eval" / ("health" if SCALE == "full" else f"health_{SCALE}")
os.environ.update({
    "SANITISE_LLM_BACKEND": "ollama", "SANITISE_MODEL": MODEL, "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid", "SANITISE_OCR_MODEL": OCR_MODEL,
    "SANITISE_AGENTS_BACKEND": "gemini", "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
    "SANITISE_GEMINI_API_KEY": key, "GEMINI_API_KEYS": key,
    "SANITISE_MONITOR_DB_PATH": str(WORK / "stores" / "monitor.sqlite"),
})
RUN = f"python tools/eval_suite/run.py {{step}} --domain health --work {WORK} --scale {SCALE} --people-file {PEOPLE_FILE}"
def run(step, extra=""):
    colab_cache.start_ollama()  # back up if it stopped (e.g. a cell was interrupted)
    cmd = RUN.format(step=step) + " " + extra
    print("$", cmd, flush=True)
    code = subprocess.run(cmd, shell=True, check=False).returncode
    if code and step != "report":  # stop here: later numbers would be wrong or missing
        raise RuntimeError(f"{step} failed: read the last lines above, then run this cell again "
                           "(every step resumes where it stopped)")
print(f"ready: answers {MODEL} | Agents 1-3 {GEMINI_MODEL} ({len(key.split(','))} key(s)) | "
      f"OCR {OCR_MODEL} | results in {WORK}")

In [ ]:
# 2 Data: real public text downloaded now, the generated corpus copied, every question written
run("data")

In [ ]:
# 3 End to end, part 1: the 50 real scans read by OCR (main's step 0), cut into parts
run("e2e-prep")

In [ ]:
# 4 End to end, part 2: Gemini marks each scan's private values and writes 2 questions after them
# and 2 about the content (saved in e2e/labels.json: open it and check a few, see the guide)
run("e2e-label")

In [ ]:
# 4b Utility questions: harmless, about the content only, the same for both pipelines
run("utility-questions")

In [ ]:
# 5 Build: Agents 1-3 over the generated corpus, the real corpus and both end-to-end sets
# (long; re-run after a time limit: documents already in a store are skipped)
run("build")

In [ ]:
# 6 Single-question attacks: Target Info, Repeat / ROUGE Prompt, membership inference, utility
run("single")

In [ ]:
# 7 Multi-question attacks: 180 sessions of 8 rounds, no privacy vs ours (with the monitor)
run("multi")

In [ ]:
# 8 End to end, part 3: the scans' questions on the gold-text store (A) and the OCR store (B)
run("e2e")

In [ ]:
# 9 Report: every metric, REPORT.md and the figures
run("report")
from IPython.display import Image, Markdown, display
display(Markdown((WORK / "results" / "REPORT.md").read_text()))
for fig in sorted((WORK / "results" / "figures").glob("*.png")):
    display(Image(str(fig)))

In [ ]:
# 10 Pack the results to download (Output tab, or Save Version) and commit to the repo
import shutil, zipfile
zip_path = W / f"eval_results_health{'' if SCALE == 'full' else '_' + SCALE}.zip"
keep = ["results", "answers", "data/targeted.json", "data/sessions.json", "data/repeat.json",
        "data/membership.json", "data/person_membership.json", "data/utility.json", "pii/labels.json", "e2e/labels.json", "e2e/manifest.json"]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for item in keep:
        p = WORK / item
        for f in ([p] if p.is_file() else sorted(p.rglob("*")) if p.exists() else []):
            if f.is_file():
                z.write(f, f"health/" + str(f.relative_to(WORK)))
print(f"{zip_path}: {zip_path.stat().st_size / 1e6:.1f} MB")
print("Download it from the Output tab (or Save Version), then follow 'Saving the results' in the guide.")